# Paper-scale Pong DQN — strict assignment baseline

This notebook reproduces the paper-scale run (10M emulator frames = 2.5M decisions, one gradient update per decision) with the **strict assignment baseline**: `paper2013` CNN (`16/32` conv, 256 hidden), ReLU, MSE loss, vanilla target-network DQN.

## Prerequisites
1. **Push the current working tree to GitHub** (`git add -A && git commit -m "assignment baseline + long-run tooling" && git push origin MyBranch`). The notebook checks out `MyBranch`; cell 3 fails loudly if the clone does not contain the assignment-baseline defaults.
2. Kaggle notebook settings: **Accelerator = GPU (T4 x2)** and **Internet = On**.
3. Expected wall clock: **~8–10 h**. Kaggle stops notebooks after a long session limit; save the artifacts at the end.

The same recipe on the pre-baseline code produced **Pong 21.0** (ten checkpoints, verified 100/100 perfect games). See `outputs/assignment-2026-10-19-kaggle-pong/REPORT.md` in the repo.


In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
import os
if not os.path.isdir('/kaggle/working/DQN'):
    !git clone https://github.com/Sokach112/DQN.git
%cd /kaggle/working/DQN
!git checkout MyBranch || echo "WARNING: MyBranch not found; using the default branch"
!git log --oneline -3
!pip install -q -r requirements.txt


In [ ]:
from omegaconf import OmegaConf
cfg = OmegaConf.load('config/config.yaml').algorithm
defaults = {k: cfg.get(k) for k in ('architecture', 'activation', 'loss', 'use_target', 'double_dqn')}
print(defaults)
assert defaults == {'architecture': 'paper2013', 'activation': 'relu', 'loss': 'mse', 'use_target': True, 'double_dqn': False}, \
    "The clone is not the assignment-baseline code: commit and push the current working tree, then rerun this cell."
print('assignment-baseline defaults confirmed')


In [ ]:
# Short pipeline smoke test: 25k decisions, 5k updates, no evaluation or checkpoints.
!python main.py total_epoch=25000 interact_per_epoch=1 \
  env.num_testing_envs=1 \
  algorithm.buffer_size=20000 algorithm.learning_starts=20000 \
  algorithm.batch_size=32 algorithm.epsilon_timestep=250000 \
  test_interval=0 save_interval=0 use_tensorboard=false \
  +save_pth=/kaggle/working/dqn_checkpoints/smoke_pong.pth 2>&1 | tail -n 20


## Main run — 2,500,000 decisions (10M frames), 2,500,000 gradient updates

Do not close the browser tab until this finishes (~8–10 h). The output is also written to `/kaggle/working/pong_paper2013_training.log`.

In [ ]:
!python main.py \
  total_epoch=2500000 interact_per_epoch=1 \
  env.num_testing_envs=1 \
  algorithm.architecture=paper2013 algorithm.activation=relu \
  algorithm.loss=mse algorithm.use_target=true algorithm.double_dqn=false \
  algorithm.buffer_size=200000 algorithm.learning_starts=20000 \
  algorithm.learning_rate=1e-4 algorithm.batch_size=32 \
  algorithm.epsilon_timestep=250000 algorithm.end_epsilon=0.1 \
  algorithm.target_update_interval=1000 \
  test_interval=50000 test_episodes=10 save_interval=100000 \
  use_tensorboard=false \
  +save_pth=/kaggle/working/dqn_checkpoints/pong_paper2013.pth \
  2>&1 | tee /kaggle/working/pong_paper2013_training.log


## Independent verification

Parses the training log and re-evaluates the saved checkpoint on **fresh reset seeds** (100 complete games), recording every score.

In [ ]:
!python kaggle_results_analysis.py \
  --log /kaggle/working/pong_paper2013_training.log \
  --checkpoint /kaggle/working/dqn_checkpoints/pong_paper2013.pth \
  --output /kaggle/working/pong_paper2013_analysis \
  --game PongNoFrameskip-v4 --games 100 --device cuda


In [ ]:
!mkdir -p /kaggle/working/artifacts
!cp /kaggle/working/pong_paper2013_training.log /kaggle/working/artifacts/
!cp /kaggle/working/dqn_checkpoints/pong_paper2013.pth /kaggle/working/artifacts/
!cp -r /kaggle/working/pong_paper2013_analysis /kaggle/working/artifacts/
!tar -czf /kaggle/working/pong_paper2013_artifacts.tgz -C /kaggle/working artifacts
!ls -lh /kaggle/working/pong_paper2013_artifacts.tgz


## Expected outcome

- Periodic greedy evaluation should climb from ≈−21 (random) through 0 around 250k decisions, >19 around 400k, and **21.0** around 800k–1M decisions if it matches the previous paper-scale run.
- The assignment baseline network is smaller than the previously successful network, so do not panic if it is slower; report the actual curve.
- Download `pong_paper2013_artifacts.tgz` from the Kaggle output panel and hand it back for integration (the log + checkpoint are enough for independent verification).